In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

Verificação das estruturas dos datasets

In [ ]:
import pandas as pd

files = [
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/codefeedback_python.parquet",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/stackoverflow_python.parquet",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/dataset_python_qa.parquet",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/code_qa_updated.parquet",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/faq_python.parquet",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/glaive_python.parquet"
]

for file in files:
    print("=" * 100)
    print(f"Arquivo: {file}")

    df = pd.read_parquet(file)

    print("\nColunas:")
    print(df.columns.tolist())

    print("\nTipos:")
    print(df.dtypes)

    print("\nPrimeiras linhas:")

    print(df.head())

Ajustando coluna de respostas do dataset_python_qa

In [ ]:
import pandas as pd
import ast
import re

df = pd.read_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/dataset_python_qa.parquet"
)

def limpar_answer(texto):
    try:
        resposta = ast.literal_eval(texto)

        if isinstance(resposta, list):
            texto = "\n".join(str(item) for item in resposta)
        else:
            texto = str(resposta)

    except (ValueError, SyntaxError, TypeError):
        texto = str(texto)

    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)

    return texto.strip()

df["answer"] = df["answer"].apply(limpar_answer)

print(df["answer"].iloc[0])

df.to_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/dataset_python_qa.parquet",
    index=False
)

print("Dataset atualizado com sucesso!")

Limpeza e padronização dos dados do dataset Codefeedback

In [ ]:
import pandas as pd
import ftfy

df = pd.read_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/codefeedback_python.parquet"
)

df["code"] = ""

df = df.dropna(subset=["question", "answer"])

for col in ["question", "answer"]:
    df[col] = (
        df[col]
        .astype(str)
        .apply(ftfy.fix_text)
    )

for col in ["question", "answer"]:
    df[col] = (
        df[col]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

df = df[
    (df["question"] != "") &
    (df["answer"] != "")
]

df = df.drop_duplicates(subset=["question", "answer"])

df = df[["question", "code", "answer", "source"]]

df.to_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/processed/codefeedback_python.parquet",
    index=False
)

print(f"Registros finais: {len(df):,}")
print("Dataset processado com sucesso!")

Limpeza e padronização dos dados do dataset da Stackoverflow

In [ ]:
import pandas as pd
import ftfy
import html
import re

df = pd.read_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/stackoverflow_python.parquet"
)

df = df.dropna(subset=["question", "answer"])

def clean_html(text):
    text = str(text)

    text = ftfy.fix_text(text)

    text = html.unescape(text)

    text = re.sub(
        r"</?(?:p|pre|code|a|div|span|br|ul|ol|li|img|strong|em|blockquote|table|tr|td|th)[^>]*>",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r"\s+", " ", text)

    return text.strip()

for col in ["question", "answer"]:
    df[col] = df[col].apply(clean_html)

df = df[
    (df["question"] != "") &
    (df["answer"] != "")
]

df = (
    df.sort_values("answer_score", ascending=False)
      .drop_duplicates(subset=["question"])
)

df["code"] = ""

df = df.drop_duplicates(subset=["question", "answer"])

df = df[["question", "code", "answer", "source"]]

df.to_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/processed/stackoverflow_python.parquet",
    index=False
)

print(f"Registros finais: {len(df):,}")
print("Dataset processado com sucesso!")

Limpeza e padronização dos dados do dataset_python_qa

In [ ]:
import pandas as pd
import ftfy
import re

df = pd.read_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/dataset_python_qa.parquet"
)

print("Registros iniciais:", len(df))

df = df.dropna(subset=["question", "answer"])

for col in ["question", "answer"]:
    df[col] = (
        df[col]
        .astype(str)
        .apply(ftfy.fix_text)
    )

for col in ["question", "answer"]:
    df[col] = (
        df[col]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

df = df[
    (df["question"] != "") &
    (df["answer"] != "")
]

df = df.drop_duplicates(
    subset=["question", "answer"]
)

df["code"] = ""

if "source" not in df.columns:
    df["source"] = "dataset_python_qa"

df = df[
    ["question", "code", "answer", "source"]
]

print("Registros finais:", len(df))

df.to_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/processed/dataset_python_qa.parquet",
    index=False
)

print("Dataset processado com sucesso!")

display(df.head())

Limpeza e padronização dos dados do dataset code_qa_updated

In [ ]:
import pandas as pd
import ftfy

df = pd.read_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/code_qa_updated.parquet"
)

print("Registros iniciais:", len(df))

df = df.dropna(
    subset=["question", "answer"]
)

for col in ["question", "answer"]:
    df[col] = (
        df[col]
        .astype(str)
        .apply(ftfy.fix_text)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

df["code"] = (
    df["code"]
    .astype(str)
    .str.strip()
)

df = df[
    (df["question"] != "") &
    (df["answer"] != "")
]

df["source"] = "code_qa_updated"

df = df.drop_duplicates(
    subset=["question", "code", "answer"]
)

df = df[
    ["question", "code", "answer", "source"]
]

print("Registros finais:", len(df))

df.to_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/processed/code_qa_updated.parquet",
    index=False
)

print("Dataset processado com sucesso!")

display(df.head())

Limpeza e padronização dos dados do dataset faq_python.parquet

In [ ]:
import pandas as pd
import ftfy

df = pd.read_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/faq_python.parquet"
)

print("Registros iniciais:", len(df))

df = df.dropna(
    subset=["question", "answer"]
)

df["code"] = ""

for col in ["question", "answer"]:
    df[col] = (
        df[col]
        .astype(str)
        .apply(ftfy.fix_text)
    )

for col in ["question", "answer"]:
    df[col] = (
        df[col]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

df = df[
    (df["question"] != "") &
    (df["answer"] != "")
]

df = df.drop_duplicates(
    subset=["question", "answer"]
)

df["source"] = "faq"

df = df[
    ["question", "code", "answer", "source"]
]

print("Registros finais:", len(df))

df.to_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/processed/faq_python.parquet",
    index=False
)

print("Dataset processado com sucesso!")

display(df.head())

Limpeza e padronização dos dados do dataset glaive_python.parquet

In [ ]:
import pandas as pd
import ftfy

df = pd.read_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/filtered/glaive_python.parquet"
)

print("Registros iniciais:", len(df))

df = df.dropna(
    subset=["question", "answer"]
)

df["code"] = ""

for col in ["question", "answer"]:
    df[col] = (
        df[col]
        .astype(str)
        .apply(ftfy.fix_text)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

df = df[
    (df["question"] != "") &
    (df["answer"] != "")
]

df = df.drop_duplicates(
    subset=["question", "answer"]
)

df["source"] = "glaive_python"

df = df[
    ["question", "code", "answer", "source"]
]

print("Registros finais:", len(df))

df.to_parquet(
    "/content/drive/MyDrive/TCC_Chatbot/datasets/processed/glaive_python.parquet",
    index=False
)

print("Dataset processado com sucesso!")

display(df.head())

Verificação final das estruturas dos datasets após limpeza e padronização dos dados

In [ ]:
import pandas as pd

files = [
    "/content/drive/MyDrive/TCC_Chatbot/datasets/processed/codefeedback_python.parquet",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/processed/stackoverflow_python.parquet",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/processed/dataset_python_qa.parquet",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/processed/code_qa_updated.parquet",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/processed/faq_python.parquet",
    "/content/drive/MyDrive/TCC_Chatbot/datasets/processed/glaive_python.parquet"
]

for file in files:
    print("=" * 100)
    print(f"Arquivo: {file}")

    df = pd.read_parquet(file)

    print("\nColunas:")
    print(df.columns.tolist())

    print("\nTipos:")
    print(df.dtypes)

    print("\nPrimeiras linhas:")

    print(df.head())